## Stress test: many deployments x many rows x many classes

Exercises `insert_deployment` / `insert_recording` / `insert_deployment_scores`
at realistic-large scale, without touching real audio files -- recordings
are registered with synthetic filenames, and score matrices are generated
in-memory per deployment (never all held at once: at
`N_ROWS_PER_DEPLOYMENT x N_CLASSES` float16 (the on-disk storage dtype;
see `ParquetScoresDB.get_embedding_dtype()`), one deployment's matrix
alone is `N_ROWS_PER_DEPLOYMENT * N_CLASSES * 2` bytes -- e.g. 100k x 10k
is 2GB, still too big to hold for all `N_DEPLOYMENTS` simultaneously).

Tune `N_DEPLOYMENTS_TO_RUN` down for a quick smoke test before committing to
the full `N_DEPLOYMENTS`.

In [1]:
import shutil
import time
from pathlib import Path

from ml_collections import config_dict
import numpy as np
from perch_hoplite.db import parquet_scores_impl as psi

from tqdm.autonotebook import tqdm

# --- Stress-test scale (exposed for tuning) ---------------------------------
N_DEPLOYMENTS = 1_000
N_ROWS_PER_DEPLOYMENT = 100_000  # scored clips per deployment
N_CLASSES = 5
RECORDINGS_PER_DEPLOYMENT = 50  # rows are spread across this many "files"
CLIP_DURATION_S = 3.0

# How many deployments to actually insert this run. Set equal to
# N_DEPLOYMENTS for the full stress test; keep small (e.g. 2-3) to sanity
# check the code path quickly before committing to a multi-GB/multi-hour run.
N_DEPLOYMENTS_TO_RUN = 1_000
STRESS_DB_PATH = Path("/tmp/parquet_scores_stress_db")

print(
    f"Full target: {N_DEPLOYMENTS} deployments x {N_ROWS_PER_DEPLOYMENT} rows x"
    f" {N_CLASSES} classes"
    f" ({N_ROWS_PER_DEPLOYMENT * N_CLASSES * 2 / 1e9:.1f} GB/deployment as"
    " float16 on disk)."
)
print(f"This run: inserting {N_DEPLOYMENTS_TO_RUN} deployment(s).")

Full target: 1000 deployments x 100000 rows x 5 classes (0.0 GB/deployment as float16 on disk).
This run: inserting 1000 deployment(s).


/var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr/T/ipykernel_33291/4129572273.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# Fresh DB. No real audio files/folders are created -- recordings are
# registered with synthetic filenames only.
db.db.close()
shutil.rmtree(STRESS_DB_PATH, ignore_errors=True)


NameError: name 'db' is not defined

In [3]:
class_labels = [f"class_{i:05d}" for i in range(N_CLASSES)]
db = psi.ParquetScoresDB.create(STRESS_DB_PATH, class_labels=class_labels)
len(db.get_all_deployments())

0

In [77]:
import datetime

In [78]:
def make_synthetic_deployment_scores(
    deployment_id: int,
    n_rows: int,
    n_classes: int,
    n_recordings: int,
    rng: np.random.Generator,
) -> tuple[list[int], list[float], list[float], np.ndarray]:
  """Generate synthetic (recording_id, start_time, end_time, scores) for one deployment.

  Rows are spread evenly across `n_recordings` synthetic recordings, each
  with consecutive non-overlapping clips -- mirroring how a real deployment's
  scored clips are laid out (many clips per file, files concatenated in
  time), without touching any real audio.
  """
  rows_per_recording = n_rows // n_recordings
  recording_ids = []
  start_times = []
  end_times = []
  for local_idx in range(n_recordings):
    recording_id = db.insert_recording(
        filename=f"deployment_{deployment_id}/synthetic_{local_idx:04d}.wav",
        datetime=datetime.datetime(2024, 1, 1, 10, 0, 0, tzinfo=datetime.timezone.utc)+ 
            datetime.timedelta(days=local_idx),
        deployment_id=deployment_id,
    )
    for clip_idx in range(rows_per_recording):
      recording_ids.append(recording_id)
      start_times.append(clip_idx * CLIP_DURATION_S)
      end_times.append((clip_idx + 1) * CLIP_DURATION_S)

  # Pad up to n_rows if n_rows isn't evenly divisible by n_recordings.
  remainder = n_rows - len(recording_ids)
  if remainder:
    recording_ids.extend([recording_ids[-1]] * remainder)
    last_end = end_times[-1]
    start_times.extend(
        last_end + i * CLIP_DURATION_S for i in range(remainder)
    )
    end_times.extend(
        last_end + (i + 1) * CLIP_DURATION_S for i in range(remainder)
    )

  scores = rng.normal(loc=-3.4, scale=0.3, size=(n_rows, n_classes)).astype(
      np.float32
  )
  return recording_ids, start_times, end_times, scores

In [ ]:
rng = np.random.default_rng(0)
insert_timings = []
scan_timings = []

for i in tqdm(range(N_DEPLOYMENTS_TO_RUN)):
  deployment_id = db.insert_deployment(
      name=f"stress_deployment_{i:05d}",
      project="stress_test",
  )

  t0 = time.perf_counter()
  recording_ids, start_times, end_times, scores = make_synthetic_deployment_scores(
      deployment_id,
      N_ROWS_PER_DEPLOYMENT,
      N_CLASSES,
      RECORDINGS_PER_DEPLOYMENT,
      rng,
  )
  t_generate = time.perf_counter() - t0

  t0 = time.perf_counter()
  db.insert_deployment_scores(
      deployment_id=deployment_id,
      recording_ids=recording_ids,
      start_times=start_times,
      end_times=end_times,
      scores=scores,
      handle_existing_recordings="overwrite",
  )
  t_write = time.perf_counter() - t0
  db.commit()

  chunk_paths = db.deployment_chunk_paths(deployment_id)
  parquet_size_mb = sum(p.stat().length for p in chunk_paths) / 1e6

  insert_timings.append((deployment_id, t_generate, t_write, parquet_size_mb))
#   print(
#       f"deployment {deployment_id}: generate={t_generate:.2f}s"
#       f" write={t_write:.2f}s parquet_size={parquet_size_mb:.1f}MB"
#   )

  # Drop the in-memory matrix immediately; don't hold N_DEPLOYMENTS_TO_RUN
  # worth of `scores` arrays alive at once.
  del recording_ids, start_times, end_times, scores

In [54]:
db.select_scores?

Signature:
db.select_scores(
    class_name: str,
    selection: Literal['top', 'random', 'binned'] = 'top',
    n: int | None = None,
    bin_edges: collections.abc.Sequence[float] | None = None,
    by_deployment: bool = False,
    date_bins: Union[Literal['day', 'week'], collections.abc.Sequence[tuple[datetime.datetime, str]], NoneType] = None,
    deployments_filter: ml_collections.config_dict.config_dict.ConfigDict | None = None,
    recordings_filter: ml_collections.config_dict.config_dict.ConfigDict | None = None,
    min_score: float | None = None,
    max_score: float | None = None,
    start_date: datetime.datetime | None = None,
    end_date: datetime.datetime | None = None,
    random_seed: float | None = None,
) -> Any
Docstring:
Select scores for a single class in one DuckDB query, no Python loop.

This is the general-purpose scores-selection entry point: it composes a
selection mode (highest-scoring, random, or percentile-binned) with
optional, independent stratification

In [8]:
target_class = class_labels[0]


In [ ]:
# Query-side stress: single-class scan across every deployment written so
# far. This is the access pattern the whole design optimizes for -- column
# projection + row-group pruning over N_DEPLOYMENTS_TO_RUN Parquet files,
# each with N_CLASSES columns.

n = 50
t0 = time.perf_counter()
top_scores = db.select_scores(
    target_class, selection="random", bin_edges=[0,.5,.75,.875,1], n=n, by_deployment=False, date_bins=None
)
t_scan = time.perf_counter() - t0
print(f"Time to select n={n} random scores on 4 score bins: {t_scan:.3f}s")


Time to select n=50 random scores on 4 score bins: 1.708s


In [97]:
# Query-side stress: single-class scan across every deployment written so
# far. This is the access pattern the whole design optimizes for -- column
# projection + row-group pruning over N_DEPLOYMENTS_TO_RUN Parquet files,
# each with N_CLASSES columns.
target_class = class_labels[0]
n = 50
t0 = time.perf_counter()
top_scores = db.select_scores(
    target_class, selection="random", bin_edges=[0,.5,.75,.875,1], n=n, by_deployment=False, date_bins=None
)
t_scan = time.perf_counter() - t0
print(f"Time to select n={n} random scores on 4 score bins for each of 1000 deployments: {t_scan:.3f}s")


Time to select n=50 random scores on 4 score bins for each of 1000 deployments: 1.632s


In [81]:
d0=datetime.datetime(2024, 1, 1, tzinfo=datetime.timezone.utc)
date_bins = [
    (d0+datetime.timedelta(days=i*2),d0 + datetime.timedelta(days=i*2+2)) for i in range(5)
]
t0 = time.perf_counter()
top_scores = db.select_scores(
    target_class, selection="top", n=n, by_deployment=True, date_bins=date_bins
)
t_scan = time.perf_counter() - t0
print(f"Time to select n={n} top scores on 5 date bins and 1000 deployments: {t_scan:.3f}s")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Time to select n=4 top scores on 5 date bins and 1000 deployments: 37.586s


In [93]:
t0 = time.perf_counter()
top_scores = db.select_scores(
    target_class, selection="top", n=n, by_deployment=False, date_bins=None, min_score=None
)
t_scan = time.perf_counter() - t0
print(f"Time to select global n={n} top scores: {t_scan:.3f}s")


Time to select global n=4 top scores: 1.666s


In [83]:
d0=datetime.datetime(2024, 1, 1, tzinfo=datetime.timezone.utc)
date_bins = [
    (d0+datetime.timedelta(days=i*2),d0 + datetime.timedelta(days=i*2+2)) for i in range(5)
]
t0 = time.perf_counter()
top_scores = db.select_scores(
    target_class, selection="top", n=n, by_deployment=True, date_bins=date_bins,min_score=-2.2
)
t_scan = time.perf_counter() - t0
print(f"Time to select n={n} top scores on 5 date bins and 1000 deployments w threshold -2.2: {t_scan:.3f}s")


Time to select n=4 top scores on 5 date bins and 1000 deployments w threshold -2.2: 0.710s


In [3]:
db.select_detections?

Object `db.select_detections` not found.


In [57]:
db.get_all_recordings(
    filter=config_dict.create(eq=dict(deployment_id=1))
)[0]

Recording(id=1, filename='deployment_1/synthetic_0000.wav', datetime=datetime.datetime(2024, 1, 1, 10, 0, tzinfo=datetime.timezone.utc), deployment_id=1)

In [44]:
db.add_detections_from_threshold(class_name=target_class, threshold=-2.2)

[6275,
 6276,
 6277,
 6278,
 6279,
 6280,
 6281,
 6282,
 6283,
 6284,
 6285,
 6286,
 6287,
 6288,
 6289,
 6290,
 6291,
 6292,
 6293,
 6294,
 6295,
 6296,
 6297,
 6298,
 6299,
 6300,
 6301,
 6302,
 6303,
 6304,
 6305,
 6306,
 6307,
 6308,
 6309,
 6310,
 6311,
 6312,
 6313,
 6314,
 6315,
 6316,
 6317,
 6318,
 6319,
 6320,
 6321,
 6322,
 6323,
 6324,
 6325,
 6326,
 6327,
 6328,
 6329,
 6330,
 6331,
 6332,
 6333,
 6334,
 6335,
 6336,
 6337,
 6338,
 6339,
 6340,
 6341,
 6342,
 6343,
 6344,
 6345,
 6346,
 6347,
 6348,
 6349,
 6350,
 6351,
 6352,
 6353,
 6354,
 6355,
 6356,
 6357,
 6358,
 6359,
 6360,
 6361,
 6362,
 6363,
 6364,
 6365,
 6366,
 6367,
 6368,
 6369,
 6370,
 6371,
 6372,
 6373,
 6374,
 6375,
 6376,
 6377,
 6378,
 6379,
 6380,
 6381,
 6382,
 6383,
 6384,
 6385,
 6386,
 6387,
 6388,
 6389,
 6390,
 6391,
 6392,
 6393,
 6394,
 6395,
 6396,
 6397,
 6398,
 6399,
 6400,
 6401,
 6402,
 6403,
 6404,
 6405,
 6406,
 6407,
 6408,
 6409,
 6410,
 6411,
 6412,
 6413,
 6414,
 6415,
 6416,
 6417,

In [ ]:
db.classes()

('class_00000', 'class_00001', 'class_00002', 'class_00003', 'class_00004')

In [49]:
db.get_all_detections()[-1]

Detection(id=9411, recording_id=49986, offsets=[555.0, 558.0], class_name='class_00000', score=-2.05859375, n=None)

In [50]:
d=db.get_detection(9411)

In [51]:
d

Detection(id=9411, recording_id=49986, offsets=[555.0, 558.0], class_name='class_00000', score=-2.05859375, n=None)

In [63]:
db.select_detections(
    class_name=None,
    selection='top',
    n=10,
    # deployments_filter=config_dict.create(eq=dict(id=1))
)

,detection_id,deployment_id,deployment_name,recording_id,recording_filename,recording_datetime,start_time,end_time,class_name,score
0,4260,336,stress_deployment_00335,16768,deployment_336/synthetic_0017.wav,2024-01-18T10:00:00+00:00,1938.0,1941.0,class_00000,-1.675781
1,7397,336,stress_deployment_00335,16768,deployment_336/synthetic_0017.wav,2024-01-18T10:00:00+00:00,1938.0,1941.0,class_00000,-1.675781
2,1123,336,stress_deployment_00335,16768,deployment_336/synthetic_0017.wav,2024-01-18T10:00:00+00:00,1938.0,1941.0,class_00000,-1.675781
3,9294,956,stress_deployment_00955,47785,deployment_956/synthetic_0034.wav,2024-02-04T10:00:00+00:00,5538.0,5541.0,class_00000,-1.750977
4,6157,956,stress_deployment_00955,47785,deployment_956/synthetic_0034.wav,2024-02-04T10:00:00+00:00,5538.0,5541.0,class_00000,-1.750977
5,3020,956,stress_deployment_00955,47785,deployment_956/synthetic_0034.wav,2024-02-04T10:00:00+00:00,5538.0,5541.0,class_00000,-1.750977
6,9285,954,stress_deployment_00953,47678,deployment_954/synthetic_0027.wav,2024-01-28T10:00:00+00:00,858.0,861.0,class_00000,-1.780273
7,3011,954,stress_deployment_00953,47678,deployment_954/synthetic_0027.wav,2024-01-28T10:00:00+00:00,858.0,861.0,class_00000,-1.780273
8,6148,954,stress_deployment_00953,47678,deployment_954/synthetic_0027.wav,2024-01-28T10:00:00+00:00,858.0,861.0,class_00000,-1.780273
9,5939,888,stress_deployment_00887,44369,deployment_888/synthetic_0018.wav,2024-01-19T10:00:00+00:00,558.0,561.0,class_00000,-1.781250


In [58]:
# get_scores_batch stress: fetch full N_CLASSES-wide vectors for a handful of
# specific keys, exercising the LEFT JOIN-by-offset path.
sample_recording = db.get_all_recordings(
    filter=config_dict.create(eq=dict(deployment_id=1))
)[0]
t0 = time.perf_counter()
batch = db.get_scores_batch(
    deployment_id=1,
    recording_ids=[sample_recording.id] * 5,
    start_times=[0.0, 3.0, 6.0, 9.0, 12.0],
    end_times=[3.0, 6.0, 9.0, 12.0, 15.0],
)
t_batch = time.perf_counter() - t0
print(f"get_scores_batch (5 keys, {N_CLASSES} classes each): {t_batch:.3f}s, shape={batch.shape}")

get_scores_batch (5 keys, 50 classes each): 0.049s, shape=(5, 50)


In [64]:
# Cleanup. Comment out to inspect the on-disk DB/Parquet files afterward.
shutil.rmtree(STRESS_DB_PATH, ignore_errors=True)